Below is the code implementing the Full Primal, Interim Allocation, BIC Full Feasibility with Borders Linear Program. Just run the following cell to initialize the code.

In [4]:
from gurobipy import Model, GRB, quicksum
import numpy as np
import itertools


def Primal_Borders(V, f):

    # Step 1: Create a new model
    model = Model("maximize_function")

    # Example dimensions for variables and vectors
    num_i = len(V[0])  # Number of bidders i
    num_j = len(V[0][0])  # Number of items j
    

    # Step 2: Define decision variables for y and p
    x = model.addVars(len(V), num_i, num_j, vtype=GRB.CONTINUOUS, name="y")  # y_i(v) as a decision variable
    p = model.addVars(len(V), num_i, vtype=GRB.CONTINUOUS, name="p") # p_i(v) as a decision variable

    # Step 3: Define the function f(v)
    # Step 4: Set the objective
    model.setObjective(
        (quicksum(quicksum(f[v][i] * quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v, i] for v in range(len(V)))
        for i in range(num_i))
    ), 
    GRB.MAXIMIZE
    )

    model.addConstrs(
      ((quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v,i] >= quicksum(x[v_p, i, j] * V[v][i][j] for j in range(num_j))- p[v_p, i])
       for v_p in range(len(V)) 
       for v in range(len(V)) 
       for not_i in range(num_i)
       for i in range(num_i)
       if not_i != i
       if np.all(V[v][not_i]==V[v_p][not_i])
       
        ), "DSIC")

    model.addConstrs((p[v, i] - quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) <= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "IR")
       
    for j in range(num_j):
        for r in range(len(V) + 1):
            for combination in itertools.combinations(enumerate(V, start=0), r):
                for n in range(num_i):
                    y = 1
                    S = [index for index, row in combination]
                    y *= 1 - quicksum(f[v][n] for v in S)
                    model.addConstr((quicksum(quicksum(f[v][i] * x[v, i, j] for v in S) for i in range(num_i))<= 1 - y
                    ), "item/bidder feasibility")
                    
                    y = 1
                    for m in range(num_i):
                        y *= 1 - quicksum(f[v][m] for v in S)

                    model.addConstr((quicksum(quicksum(f[v][i] * x[v, i, j] for v in S) for i in range(num_i)) <= 1 - y     
                    ), "item/bidder feasibility")
                        

    
    model.addConstrs((x[v, i, j] >= 0
                      for v in range(len(V))
                      for j in range(num_j)
                      for i in range(num_i)
                 ), "nonneg_y")

    model.addConstrs((p[v, i] >= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "nonneg_p")

    # Step 6: Optimize the model
    model.optimize()

    # Step 7: Print results
    if model.status == GRB.OPTIMAL:
        print(f"Optimal objective value: {model.objVal}")
        for v in range(len(V)):
            for i in range(num_i):
                for j in range(num_j):
                    print(f"x[{v},{i},{j}] = {x[v, i, j].X}, p[{v},{i}] = {p[v, i].X}")
    else:
        return("No optimal solution found.")
    return

Below are the parameters that we will enter into our linear program. They are currently set to have the type space be uniform [1,2] for a two bidder two item scenario. V contains the valuations for the bidders. f is the distribution of each v in V. f should sum to 1. It currently represents a uniform distribution. To solve the linear program with these parameters, just run the cell below. Reading the output of the program, the Optimal objective value will display the optimal output from the parameters. The display will then show each valuation and the allocation they receive. For example, y[3,0,1] = 1.0 means that in the fourth v (we start counting from 0), the first bidder (bidder 0) has an allocation of 1.0 for the second item (item 1).

In [5]:
V=[[[1,2],[1,2]],
   [[1,2],[1,1]],
   [[1,2],[2,2]],
   [[1,2],[2,1]],
  [[1,1],[1,2]],
  [[1,1],[1,1]],
  [[1,1],[2,2]],
  [[1,1],[2,1]],
    [[2,2],[1,2]],
    [[2,2],[1,1]],
    [[2,2],[2,2]],
    [[2,2],[2,1]],
    [[2,1],[1,2]],
    [[2,1],[1,1]],
    [[2,1],[2,2]],
    [[2,1],[2,1]]]

# v matrices
#f = [1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16]  #distribution for which v is being chosen

f = [[1/16, 1/16] for _ in range(16)]


Primal_Borders(V,f)

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 23.6.0 23G93)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 524544 rows, 96 columns and 8389376 nonzeros
Model fingerprint: 0x62d0c8c3
Coefficient statistics:
  Matrix range     [6e-02, 2e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-02, 1e+00]
Presolve removed 393346 rows and 0 columns
Presolve time: 3.96s
Presolved: 96 rows, 131198 columns, 2097824 nonzeros

Concurrent LP optimizer: dual simplex and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.424e+03
 Factor NZ  : 2.160e+03 (roughly 50 MB of memory)
 Factor Ops : 6.818e+04 (less than 1 second per iteration)
 Threads    : 3

Barrier performed 0 iterations in 4.34 seconds (5.48 work units)
Barrier solve interrupted - model solved by another algorithm


Solved wi

This next cell attempts to separate the border's constraint into item and bidder feasibility. It essentially implements the border's constraint only for complete sets of valuations (where there exists valuations for all bidders for all items) as well as traditional item feasibility. Run the cell below to initialize this code.

In [6]:
import numpy as np
import itertools


def Primal_Borders_Sep(V, f):

    # Step 1: Create a new model
    model = Model("maximize_function")

    # Example dimensions for variables and vectors
    num_i = len(V[0])  # Number of bidders i
    num_j = len(V[0][0])  # Number of items j
    

    # Step 2: Define decision variables for y and p
    x = model.addVars(len(V), num_i, num_j, vtype=GRB.CONTINUOUS, name="y")  # y_i(v) as a decision variable
    p = model.addVars(len(V), num_i, vtype=GRB.CONTINUOUS, name="p") # p_i(v) as a decision variable

    # Step 3: Define the function f(v)
    # Step 4: Set the objective
    model.setObjective(
        (quicksum(quicksum(f[v][i] * quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v, i] for v in range(len(V)))
        for i in range(num_i))
    ), 
    GRB.MAXIMIZE
    )

    model.addConstrs(
      ((quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v,i] >= quicksum(x[v_p, i, j] * V[v][i][j] for j in range(num_j))- p[v_p, i])
       for v_p in range(len(V)) 
       for v in range(len(V)) 
       for not_i in range(num_i)
       for i in range(num_i)
       if not_i != i
       if np.all(V[v][not_i]==V[v_p][not_i])
       
        ), "DSIC")

    model.addConstrs((p[v, i] - quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) <= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "IR")
       
    for j in range(num_j):
        for r in range(len(V) + 1):
            for combination in itertools.combinations(enumerate(V, start=0), r):
                y = 1
                S = [index for index, row in combination] 
                for m in range(num_i):
                    y *= 1 - quicksum(f[v][m] for v in S)

                model.addConstr((quicksum(quicksum(f[v][i] * x[v, i, j] for v in S) for i in range(num_i)) <= 1 - y     
                ), "item/bidder feasibility")
    
    
    model.addConstrs((quicksum(x[v, i, j] for i in range(num_i)) <= 1
                      for v in range(len(V))
                      for j in range(num_j)
                 ), "item feasibility")

    
    model.addConstrs((x[v, i, j] >= 0
                      for v in range(len(V))
                      for j in range(num_j)
                      for i in range(num_i)
                 ), "nonneg_y")

    model.addConstrs((p[v, i] >= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "nonneg_p")

    # Step 6: Optimize the model
    model.optimize()

    # Step 7: Print results
    if model.status == GRB.OPTIMAL:
        print(f"Optimal objective value: {model.objVal}")
        for v in range(len(V)):
            for i in range(num_i):
                for j in range(num_j):
                    print(f"x[{v},{i},{j}] = {x[v, i, j].X}, p[{v},{i}] = {p[v, i].X}")
    else:
        return("No optimal solution found.")
    return

Run the cell below to execute the linear program with the seperated border's constraint.

In [7]:
Primal_Borders_Sep(V,f)

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 23.6.0 23G93)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 131360 rows, 96 columns and 2097984 nonzeros
Model fingerprint: 0x75c8ce44
Coefficient statistics:
  Matrix range     [6e-02, 2e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 1e+00]
Presolve removed 162 rows and 0 columns
Presolve time: 1.38s
Presolved: 96 rows, 131198 columns, 2097824 nonzeros

Concurrent LP optimizer: dual simplex and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.424e+03
 Factor NZ  : 2.160e+03 (roughly 50 MB of memory)
 Factor Ops : 6.818e+04 (less than 1 second per iteration)
 Threads    : 3

Barrier performed 0 iterations in 1.74 seconds (1.95 work units)
Barrier solve interrupted - model solved by another algorithm


Solved with 